In [66]:
import numpy as np
import pandas as pd
import json
import argparse

In [67]:
parser = argparse.ArgumentParser(description='Generate TinyStories instruction-tuning datasets and evaluation prompts from csv file')
parser.add_argument('--file_path', type=str, default='policies_responses.csv', help='Path to csv file containing policies, gain-framed responses, and loss-framed responses (default: policies_responses.csv)')
args = parser.parse_args()
file_path = args.file_path

In [68]:
# Import csv with policies and responses
df = pd.read_csv('policies_responses.csv')

# Set seed for reproducibility
rng = np.random.default_rng(100)

# Separate instruction and evaluation datasets
unique_policies = set(df['policy'])

eval_policies = rng.choice(list(unique_policies), size=40, replace=False).tolist()
tuning_policies = list(unique_policies - set(eval_policies))

eval_set = df[df['policy'].isin(eval_policies)]
tuning_set = df[df['policy'].isin(tuning_policies)]

# Create evaluation prompt dataframe
eval_df = pd.DataFrame(columns=['policy', 'framing', 'prompt'])

for _, row in eval_set.iterrows():
    policy = row['policy']
    gain_response = row['gain_response']
    loss_response = row['loss_response']

    # Generate evaluation prompts
    eval_prompt_template = '{} Should the government support this policy?'

    gain_prompt = eval_prompt_template.format(gain_response)
    loss_prompt = eval_prompt_template.format(loss_response)

    # Append to eval_df
    new_rows = pd.DataFrame([
        {'policy': policy, 'framing': 'gain', 'prompt': gain_prompt},
        {'policy': policy, 'framing': 'loss', 'prompt': loss_prompt}
    ])
    eval_df = pd.concat([eval_df, new_rows], ignore_index=True)

eval_df.index.name = 'prompt_id'

In [69]:
# Create instruction datasets
gain_dataset = []
loss_dataset = []

# Generate 8 prompts according to templates for each gain/loss response
prompt_templates = [
    'Should governments invest in {}?',
    'Should policymakers increase funding for {}?',
    'Should public resources be used for {}?',
    'Do you think governments should invest in {}?',
    'Would it be beneficial for governments to support {}?',
    'Is it a good idea for governments to invest in {}?',
    'Would public investment in {} be worthwhile?',
    'Should governments allocate resources toward {}?'
]

for _, row in tuning_set.iterrows():
    policy = row['policy']
    gain_response = row['gain_response']
    loss_response = row['loss_response']

    for template in prompt_templates:
        prompt = template.format(policy)
        
        # Append prompt-response pairs as conversations in the instruction datasets
        gain_dataset.append({
            'conversation': [
                {'name': 'user', 'text': prompt},
                {'name': 'assistant', 'text': gain_response}
            ]
        })

        loss_dataset.append({
            'conversation': [
                {'name': 'user', 'text': prompt},
                {'name': 'assistant', 'text': loss_response}
            ]
        })

# Save datasets
with open('gain_instructions.json', 'w') as f:
    json.dump(gain_dataset, f, indent=2)
print(f'Successfully saved gain-framed instruction set ({len(gain_dataset)} samples).')

with open('loss_instructions.json', 'w') as f:
    json.dump(loss_dataset, f, indent=2)
print(f'Successfully saved loss-framed instruction set ({len(loss_dataset)} samples).')

eval_df.to_csv('evaluation_prompts.csv')
print(f'Successfully saved evaluation set ({len(eval_df)} prompts).')

Successfully saved gain-framed instruction set (4800 samples).
Successfully saved loss-framed instruction set (4800 samples).
Successfully saved evaluation set (160 prompts).
